# LightGBM Forecasting - Upgraded Version

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgm
import sklearn.metrics
import warnings
warnings.filterwarnings('ignore')

In [ ]:
data = pd.read_csv(r"D:\Documents\DEPI\DATA\data\train_before_2016.csv")
data.drop(columns=["Unnamed: 0"], inplace=True, errors="ignore")
data["date"] = pd.to_datetime(data["date"])
data = data.sort_values(["item_id", "store_id", "date"]).reset_index(drop=True)
print(data.shape)
print(data.columns.tolist())

## Feature Engineering

In [ ]:
# Time features
data["dayofweek"]      = data["date"].dt.dayofweek
data["week"]           = data["date"].dt.isocalendar().week.astype(int)
data["month"]          = data["date"].dt.month
data["year"]           = data["date"].dt.year
data["is_weekend"]     = data["dayofweek"].isin([5, 6]).astype(int)
data["is_month_start"] = data["date"].dt.is_month_start.astype(int)
data["is_month_end"]   = data["date"].dt.is_month_end.astype(int)

# Lag features
grp = data.groupby(["item_id", "store_id"])["sales"]
for lag in [1, 7, 14, 28]:
    data[f"lag_{lag}"] = grp.shift(lag)

# Rolling statistics
for window in [7, 14, 28]:
    data[f"rolling_mean_{window}"] = grp.transform(
        lambda x: x.shift(1).rolling(window).mean()
    )
    data[f"rolling_std_{window}"] = grp.transform(
        lambda x: x.shift(1).rolling(window).std()
    )

# Exponential weighted mean
data["ewm_7"] = grp.transform(lambda x: x.shift(1).ewm(span=7).mean())

# Optional: sell_price features
if "sell_price" in data.columns:
    data["price_momentum"] = data.groupby(["item_id","store_id"])["sell_price"].pct_change()
    data["price_rolling_mean_28"] = data.groupby(["item_id","store_id"])["sell_price"]\
        .transform(lambda x: x.rolling(28).mean())
    data["price_vs_mean"] = data["sell_price"] / (data["price_rolling_mean_28"] + 1e-8)

# Fill NaN
lag_roll_cols = [c for c in data.columns if c.startswith(("lag_","rolling_","ewm_","price_"))]
data[lag_roll_cols] = data[lag_roll_cols].fillna(0)
print("New features:", lag_roll_cols)

In [ ]:
# Categorical features
cat_cols = ["item_id","store_id","dept_id","cat_id","state_id",
            "event_name_1","event_type_1","event_name_2","event_type_2"]
cat_cols = [c for c in cat_cols if c in data.columns]

data[cat_cols] = data[cat_cols].fillna("None")
for col in cat_cols:
    data[col] = data[col].astype("category")

# d column
if "d" in data.columns and data["d"].dtype == object:
    data["d"] = data["d"].str.replace("d_", "").astype(int)

data = data.drop(columns=["date", "weekday"], errors="ignore")

## Train / Test Split (Time-based)

In [ ]:
# Time-based split - NO shuffle (كان بيعمل data leakage)
if "d" in data.columns:
    cutoff = int(data["d"].quantile(0.8))
    train = data[data["d"] <= cutoff]
    test  = data[data["d"] >  cutoff]
else:
    n = len(data)
    train = data.iloc[:int(n * 0.8)]
    test  = data.iloc[int(n * 0.8):]

print(f"Train: {len(train):,} rows")
print(f"Test:  {len(test):,} rows")

In [ ]:
x_train = train.drop("sales", axis=1).replace([np.inf, -np.inf], 0)
y_train = train["sales"].astype(float)
x_test  = test.drop("sales", axis=1).replace([np.inf, -np.inf], 0)
y_test  = test["sales"].astype(float)

## Model Training

In [ ]:
model = lgm.LGBMRegressor(
    n_estimators      = 2000,
    learning_rate     = 0.05,
    max_depth         = 8,
    num_leaves        = 127,
    min_child_samples = 20,
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    reg_alpha         = 0.1,
    reg_lambda        = 0.1,
    n_jobs            = -1,
    random_state      = 42,
)

model.fit(
    x_train, y_train,
    eval_set  = [(x_test, y_test)],
    callbacks = [lgm.early_stopping(50), lgm.log_evaluation(200)],
)
print(f"Best iteration: {model.best_iteration_}")

## Evaluation

In [ ]:
y_pred = np.clip(model.predict(x_test), 0, None)

mae  = sklearn.metrics.mean_absolute_error(y_test, y_pred)
rmse = sklearn.metrics.root_mean_squared_error(y_test, y_pred)
r2   = sklearn.metrics.r2_score(y_test, y_pred)

print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R2   : {r2:.4f}")

## Feature Importance

In [ ]:
import matplotlib.pyplot as plt

feat_imp = pd.Series(model.feature_importances_, index=x_train.columns)
feat_imp = feat_imp.sort_values(ascending=False).head(20)

plt.figure(figsize=(10, 6))
feat_imp.plot(kind='barh')
plt.title('Top 20 Feature Importances')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()